# NB1: MediCS Framework — Environment Setup

**Purpose:** Install dependencies, configure API keys, verify GPU access, and validate HuggingFace Llama-3 access.

**Run this notebook once** at the start of each Colab session. After running Cell 1 (package installation), **restart the runtime**, then run from Cell 2 onwards.

---

## Cell 1: Install Dependencies
**IMPORTANT:** After running this cell, go to Runtime → Restart runtime, then skip this cell and run from Cell 2.

In [ ]:
# ============================================================
# Cell 1: Install all dependencies (RESTART RUNTIME AFTER THIS)
# ============================================================
!pip install -q \
    transformers==4.46.3 \
    peft==0.13.2 \
    accelerate==1.1.1 \
    trl==0.12.1 \
    bitsandbytes==0.44.1 \
    datasets==3.1.0 \
    huggingface-hub==0.26.2 \
    safetensors==0.4.5 \
    sentencepiece \
    protobuf \
    openai>=1.30.0 \
    deep-translator==1.11.4 \
    tenacity>=8.2.0 \
    scipy \
    scikit-learn \
    matplotlib \
    seaborn \
    pandas \
    pyyaml \
    cryptography \
    bertviz \
    tqdm

print("\n" + "="*60)
print("INSTALLATION COMPLETE")
print(">>> RESTART RUNTIME NOW <<<")
print("Then skip this cell and run from Cell 2.")
print("="*60)

In [ ]:
# ============================================================
# Cell 1b (Optional): Install Unsloth for 2x faster LoRA training
# Only needed before running NB5_Defense.
# If this fails, NB5 will fall back to standard PEFT.
# ============================================================
# Uncomment the lines below to install Unsloth:
# !pip install --no-deps bitsandbytes accelerate xformers peft trl triton \
#     cut_cross_entropy unsloth_zoo
# !pip install --no-deps unsloth
# print("Unsloth installed. Restart runtime if needed.")

## Cell 2: Mount Google Drive & Initialize MediCS

In [ ]:
# ============================================================
# Cell 2: Mount Drive and import shared utilities
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

import sys
sys.path.insert(0, '/content/drive/MyDrive/MediCS/utils')
from medics_utils import *

config = init_medics(mount_drive=False)  # Already mounted above
print("MediCS utilities loaded.")

## Cell 3: GPU Smoke Test

In [ ]:
# ============================================================
# Cell 3: Verify GPU availability and specs
# ============================================================
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    print(f"VRAM: {props.total_mem / 1e9:.1f} GB")
    print(f"Compute capability: {props.major}.{props.minor}")
    print(f"CUDA version: {torch.version.cuda}")
    
    # Determine recommended dtype
    if props.major >= 8:  # A100, A10G, etc.
        print("\nRecommended dtype: bfloat16 (A100/Ampere+)")
        print("Flash Attention 2: SUPPORTED")
    else:  # V100, T4, etc.
        print("\nRecommended dtype: float16 (V100/Turing)")
        print("Flash Attention 2: NOT supported (will use default attention)")
else:
    print("\nWARNING: No GPU detected!")
    print("Go to Runtime → Change runtime type → GPU (T4/A100)")

## Cell 4: API Key Setup

**First time only:** Run Cell 4a to save your keys (encrypted).  
**Every session:** Run Cell 4b to load your keys.

In [ ]:
# ============================================================
# Cell 4a: FIRST TIME ONLY — Save API keys (encrypted)
# After running this once, you never need to run it again.
# Delete the key values from this cell after saving!
# ============================================================

# Uncomment and fill in your keys, then run once:
# save_api_keys({
#     "openai_api_key": "sk-...",
#     "hf_token": "hf_..."
# })

In [ ]:
# ============================================================
# Cell 4b: Load API keys (every session)
# ============================================================
keys = load_api_keys()
openai_client = setup_api_clients(keys)
print("API keys loaded and clients configured.")
print(f"OpenAI key: ...{keys['openai_api_key'][-4:]}")
print(f"HF token: ...{keys['hf_token'][-4:]}")

## Cell 5: Validate HuggingFace Llama-3 Access

In [ ]:
# ============================================================
# Cell 5: Verify HuggingFace token and Llama-3 access
# ============================================================
from huggingface_hub import login, model_info

login(token=keys["hf_token"])

MODEL_ID = "meta-llama/Meta-Llama-3-8B-Instruct"
try:
    info = model_info(MODEL_ID)
    print(f"Model accessible: {info.id}")
    print(f"Model size: {info.safetensors.total if info.safetensors else 'unknown'}")
    print("\nLlama-3 access VERIFIED.")
except Exception as e:
    print(f"ERROR: Cannot access {MODEL_ID}")
    print(f"Reason: {e}")
    print("\nFix: Go to https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct")
    print("and accept the license agreement with the same HF account.")

## Cell 6: Validate OpenAI API

In [ ]:
# ============================================================
# Cell 6: Test OpenAI API connection
# ============================================================
try:
    response = openai_client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "user", "content": "Say 'MediCS API test OK' and nothing else."}],
        max_tokens=20,
    )
    print(f"OpenAI response: {response.choices[0].message.content}")
    print(f"Model used: {response.model}")
    print("\nOpenAI API VERIFIED.")
except Exception as e:
    print(f"ERROR: OpenAI API call failed.")
    print(f"Reason: {e}")
    print("\nCheck your API key and billing at https://platform.openai.com")

## Cell 7: Verify Translation Pipeline

In [ ]:
# ============================================================
# Cell 7: Test the translation pipeline
# ============================================================
test_word = "medicine"
test_languages = ["zu", "bn", "sw"]

print(f"Testing translation of '{test_word}':")
for lang in test_languages:
    result = translate_with_fallback(test_word, source="en", target=lang)
    print(f"  {LANGUAGE_NAMES.get(lang, lang)}: {result['translation']} (via {result['source']})")
    time.sleep(0.5)

flush_translation_cache()
print("\nTranslation pipeline VERIFIED.")

## Cell 8: Version Check Summary

In [ ]:
# ============================================================
# Cell 8: Print all key library versions
# ============================================================
import transformers, peft, accelerate, trl, bitsandbytes, datasets

versions = {
    "transformers": transformers.__version__,
    "peft": peft.__version__,
    "accelerate": accelerate.__version__,
    "trl": trl.__version__,
    "bitsandbytes": bitsandbytes.__version__,
    "datasets": datasets.__version__,
    "torch": torch.__version__,
}

print("Library versions:")
for lib, ver in versions.items():
    print(f"  {lib}: {ver}")

print("\n" + "="*60)
print("NB1 SETUP COMPLETE — Ready to proceed to NB2.")
print("="*60)